# 02 — CatBoost 단일 회귀 (raw mode — 전처리 없음)

`zit_only_raw` 패턴을 일반 트리에 적용. `preprocess.run()` 생략하고 CatBoost의 NaN 네이티브 처리(`nan_mode='Min'` 기본)에 맡겨 원본 분포 그대로 학습.

- **입력**: `0_data/compet_xs_data.csv`, `compet_ys_*_data.csv`
- **출력**: `4_output/02_reg_single/catboost_raw/{best_params.json, fold_models.pkl, optuna_*.db, oof|val|test_die.csv, oof|val|test_unit.csv}`
- **전처리**: Stage 0 (`EXCLUDE_COLS` 웨이퍼맵 사전 제외)만 적용. cleaning / imputation / outlier winsorize / 상관 제거 전부 SKIP
- **HPO**: `N_TRIALS=10000` + `TIMEOUT_SEC=24h` 안전망 (런스크립트에서 타임아웃 컨트롤)
- **anchor**: 전처리 버전 anchor를 시작점으로 enqueue (feature space가 달라 raw best는 다를 수 있음)
- **손실함수**: `RMSE / Poisson / Tweedie` 3종 (`models.catboost_space()`)
- **target transform**: `'none'` 고정

## 1. 환경 설정 + 모듈 import

In [ ]:
import os, sys

# Google Drive 파일 ID들 — Colab에서 코드/데이터/모듈 zip을 자동으로 받아 풀 때 사용 (로컬은 무시)
GDRIVE_CODE_ID          = '1AD4PDBnDVjp-LSna6puB7qLnpBqB7j_I'   # code.zip = setup.py + utils/
GDRIVE_DATASET_ID       = '1yOUo0_wPLcuZBSJIK592b00YkSIlk4zO'   # dataset.zip = 원본 CSV 4개
GDRIVE_PREPROCESSING_ID = '1Rh0ByOS4Gama8XHuvY7KkOHo278H9YLr'   # preprocessing.zip (raw는 EXCLUDE_COLS·meta_features만 사용, import 호환용)
GDRIVE_MODELING_ID      = '1Vrn5LBl611rWbag7d09LZH68_lfpu6wP'   # modeling.zip = 3_modeling/modules (코드 수정 시 재업로드)
GDRIVE_OUTPUT_ID        = '1ts73qEMmjX8cKIb-QeDQ-TMeyudFGWzs'   # 4_output.zip = 기존 실험 산출물 (RESUME 시 복원용)
RESUME                  = True   # True=기존 optuna db에 trial 이어 붙임 / False=처음부터 (db 있으면 의도적 에러)

# Colab이면 필요한 zip들을 받아 풀고(이미 풀려 있으면 skip), 로컬이면 ../../../setup.py만 실행
try:
    import google.colab
    from google.colab import drive
    drive.mount('/content/drive')
    if not os.path.exists('/content/project/setup.py'):
        os.system('pip install -q gdown')
        os.system(f'gdown {GDRIVE_CODE_ID} -O /content/code.zip')
        os.system('unzip -qo /content/code.zip -d /content/project')
        os.makedirs('/content/project/0_data', exist_ok=True)
        os.system(f'gdown {GDRIVE_DATASET_ID} -O /content/project/0_data/dataset.zip')
        os.system('unzip -qo /content/project/0_data/dataset.zip -d /content/project/0_data')
        os.remove('/content/project/0_data/dataset.zip')
    if not os.path.exists('/content/project/2_preprocessing/cleaning.py'):
        os.system(f'gdown {GDRIVE_PREPROCESSING_ID} -O /content/preprocessing.zip')
        os.system('unzip -qo /content/preprocessing.zip -d /content/project')
    if not os.path.exists('/content/project/3_modeling/modules/hpo.py'):
        assert GDRIVE_MODELING_ID, 'GDRIVE_MODELING_ID가 비어있음 — modules.zip Drive ID 입력 필요'
        os.makedirs('/content/project/3_modeling', exist_ok=True)
        os.system(f'gdown {GDRIVE_MODELING_ID} -O /content/modules.zip')
        os.system('unzip -qo /content/modules.zip -d /content/project/3_modeling')
    # RESUME이면 이전 4_output을 통째로 복원 (이미 폴더 있으면 skip)
    if RESUME and GDRIVE_OUTPUT_ID and not os.path.exists('/content/project/4_output/01_zit'):
        os.system(f'gdown {GDRIVE_OUTPUT_ID} -O /content/4_output.zip')
        os.system('unzip -qo /content/4_output.zip -d /content/project')
        os.remove('/content/4_output.zip')
    sys.path.insert(0, '/content/project')
    %run /content/project/setup.py
except ImportError:
    %run ../../../setup.py

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# utils.config: 환경 공통 상수 (경로·컬럼명·SEED). utils.data: CSV 로드·split 분리.
from utils.config import PROJECT_ROOT, SEED, TARGET_COL, KEY_COL, OUTPUT_DIR
from utils.data import load_all, get_feat_cols, split_xs

# 전처리 모듈(2_preprocessing) 경로 추가 — raw는 EXCLUDE_COLS·meta_features만 쓰지만 같은 폴더라 sys.path 추가 필요
# sys.path에 2_preprocessing·3_modeling 추가 → EXCLUDE_COLS import + `from modules import` 가능.
PREP_ROOT = os.path.join(PROJECT_ROOT, '2_preprocessing')
if PREP_ROOT not in sys.path:
    sys.path.insert(0, PREP_ROOT)
# `from modules import ...`가 3_modeling/modules를 찾게
MODEL_ROOT = os.path.join(PROJECT_ROOT, '3_modeling')
if MODEL_ROOT not in sys.path:
    sys.path.insert(0, MODEL_ROOT)

# raw 모드: preprocess.run()은 사용 안 함 — EXCLUDE_COLS(Stage 0 제외 목록)만 가져옴
from modules.preprocess import EXCLUDE_COLS as _WAFER_MAP_EXCLUDE
from modules import hpo, models   # noqa: E402  (hpo.run_hpo/refit_best/save_artifacts, models 레지스트리)
from meta_features import add_meta_features   # die_xy / position 메타피처 헬퍼

print(f'PROJECT_ROOT = {PROJECT_ROOT}')
print(f'Available models: {models.AVAILABLE_MODELS}')

## 2. 실험 설정

In [ ]:
# 모델 고정 (이 노트북은 CatBoost 단일 raw)
MODEL_NAME = 'catboost'
EXP_ID     = f'reg-{MODEL_NAME}-raw-001'                # 출력 폴더/DB 파일명에 들어감 (raw임을 명시)
EXP_MEMO   = 'Raw mode — preprocess.run 생략, NaN 네이티브 처리(nan_mode=Min) + anchor enqueue'
USER       = 'jh'

# Optuna 예산 — 로컬 런스크립트에서 타임아웃으로 컨트롤. trial은 크게, timeout은 안전망
N_TRIALS         = 10000    # 외부 런스크립트 타임아웃이 실제 종료를 결정하므로 충분히 크게 (CatBoost는 학습이 느려 catboost는 시간 우선 배분 권장)
N_FOLDS          = 5
N_STARTUP_TRIALS = 40       # TPE가 학습을 시작하기 전 무작위 trial 수 (CatBoost는 trial당 비용이 커서 lgbm/xgb보다 작게)

N_JOBS      = -1               # 모델 학습 병렬도 (CatBoost는 thread_count로 매핑됨)
TIMEOUT_SEC = 24 * 60 * 60     # 24h 안전망 — 노트북 단독 실행 시 보호장치, 실제 컨트롤은 외부 런스크립트

TARGET_TRANSFORM = 'none'      # 트리는 'none' 통일 (log1p와 사실상 동등)
CLIP_Y_EXTREME   = True        # train y의 max(=1.0, 단 1건)를 두 번째로 큰 값으로 clip

# 출력 경로 — 모델명 뒤에 _raw 표기 (기존 비-raw 디렉토리와 충돌 방지)
OUT_DIR = os.path.join(OUTPUT_DIR, '02_reg_single', MODEL_NAME, 'raw', EXP_ID.split('-')[-1])
DB_PATH = os.path.join(OUT_DIR, f'optuna_{USER}_{EXP_ID}.db')   # study가 여기에 자동 저장 (RESUME 시 여기서 이어서)
os.makedirs(OUT_DIR, exist_ok=True)

# raw 모드 — PP_FIXED 없음 (cleaning/imputation/outlier 전부 SKIP, Stage 0만 적용)

# anchor — 전처리 버전 best HP (1차 OOF=0.005523). raw에서는 feature space가 달라 trial 0 시작점으로만 사용 (Optuna가 재탐색)
# CATBOOST_ANCHOR: warm-start용 HP 박제값. 전처리 버전 1차 best → raw feature space에서 탐색 시작점으로 사용.
CATBOOST_ANCHOR = {
    'loss_function':       'RMSE',
    'iterations':          1269,
    'learning_rate':       0.02712,
    'depth':               9,
    'l2_leaf_reg':         22.96,
    'random_strength':     0.1593,
    'bagging_temperature': 0.7136,
    'border_count':        183,
    'rsm':                 0.7349,
}

print(f'EXP: {EXP_ID} | USER: {USER} | raw_mode=True')
print(f'N_TRIALS={N_TRIALS}, N_FOLDS={N_FOLDS}, N_JOBS={N_JOBS}, TIMEOUT_SEC={TIMEOUT_SEC}')
print(f'TARGET_TRANSFORM={TARGET_TRANSFORM} | CLIP_Y_EXTREME={CLIP_Y_EXTREME}')
print(f'OUT_DIR={OUT_DIR}')

## 3. 데이터 로드 + target clip + transform

In [3]:
# 데이터 로드: xs(die-level) + ys dict(train/validation/test unit-level). split_xs: xs → split별 dict.
xs, ys = load_all()
feat_cols = get_feat_cols(xs)
xs_dict = split_xs(xs)
print(f'xs: {xs.shape}, feat_cols: {len(feat_cols)}')

# train y의 극단값(1.0, 1건)만 두 번째로 큰 값으로 clip — 학습 입력 안정화 (원본 ys는 보존)
# ys_input 복사본으로 원본 보존 → 재실행해도 누적 clip 없음.
ys_input = {k: v.copy() for k, v in ys.items()}
if CLIP_Y_EXTREME:
    y_raw = ys_input['train'][TARGET_COL]
    second_max = y_raw[y_raw < y_raw.max()].max()
    n_clipped = (y_raw >= 1.0).sum()
    ys_input['train'][TARGET_COL] = y_raw.clip(upper=second_max)
    print(f'[CLIP_Y_EXTREME] 1.0 → {second_max:.6f} clip, {n_clipped}개')

# 트리 모델은 target 변환이 결과를 거의 안 바꿔서 'none'으로 고정 (transform 함수 둘 다 None)
# 트리는 target 변환 없음 ('none'). LightGBM은 raw scale에서 직접 Poisson/Tweedie 목적함수 사용.
target_transform_fn = None
target_inverse_fn   = None
print(f'[target transform] {TARGET_TRANSFORM} (strategy_common §24 — 트리 target_transform=none 통일)')

[load_xs] all-NaN 행 407개 제거 → 174,573행


[load_xs] 4 position 미만 unit 1개 제거 (split별: {'train': 1}) → die 174,573 → 174,572


[load_ys] train: xs에 없는 unit 60개 제거 → 26,187
[load_ys] validation: xs에 없는 unit 22개 제거 → 8,727
[load_ys] test: xs에 없는 unit 20개 제거 → 8,729
Xs: (174572, 1091)  |  Ys: train=26,187, val=8,727, test=8,729


xs: (174572, 1091), feat_cols: 1087
[CLIP_Y_EXTREME] 1.0 → 0.097417 clip, 1개
[target transform] none (strategy_common §24 — 트리 target_transform=none 통일)


## 4. Stage 0 (웨이퍼맵 사전 제외) + 메타피처만 적용 (raw mode)

`preprocess.run()` 호출 없음. cleaning / spatial imputation / winsorize / 고상관 제거 전부 SKIP. CatBoost는 `nan_mode='Min'`(기본)으로 NaN을 자동 처리하므로 imputation 불필요.

In [ ]:
# Stage 0만 적용: 웨이퍼맵 수동 제외 리스트로 feature 사전 제거
n_before = len(feat_cols)
feat_cols_raw = [c for c in feat_cols if c not in _WAFER_MAP_EXCLUDE]
print(f'[Stage 0] 웨이퍼맵 사전 제외: {n_before} → {len(feat_cols_raw)} ({n_before - len(feat_cols_raw)}개 제거)')
print('[raw mode] cleaning / imputation / outlier winsorize / 상관 제거 전부 SKIP')
print('  CatBoost NaN 네이티브 처리 (nan_mode=Min), 원본 feature 분포 그대로 유지')

# xs_dict의 split별 DataFrame을 그대로 복사 (NaN/이상치/스케일 그대로)
xs_train = xs_dict['train'].copy()
xs_val   = xs_dict['validation'].copy()
xs_test  = xs_dict['test'].copy()

# 메타피처: 트리는 position을 raw 정수로, die_x/die_y를 연속형으로 추가 (NaN 발생 없음)
feat_cols_clean = add_meta_features(
    xs_train, xs_val, xs_test, feat_cols_raw,
    position_mode='raw', use_die_xy=True,
)

# 학습 입력 NaN 분포 sanity check
nan_pct_train = xs_train[feat_cols_clean].isna().to_numpy().mean() * 100
print(f'\n[전처리 완료] feat_cols: {len(feat_cols_clean)}')
print(f'  train: {xs_train.shape}, val: {xs_val.shape}, test: {xs_test.shape}')
print(f'  NaN in train[feat_cols]: {xs_train[feat_cols_clean].isna().to_numpy().sum():,} ({nan_pct_train:.2f}%) — CatBoost가 직접 처리')

## 5. Optuna HPO (anchor 첫 trial enqueue + wide range)

In [ ]:
# study에 박제할 재현성 메타 — 어떤 조건으로 학습됐는지 study DB만 보고도 알 수 있게
study_meta_for_save = {
    'exp_id':              EXP_ID,
    'exp_memo':            EXP_MEMO,
    'user':                USER,
    'model_name':          MODEL_NAME,
        # raw_mode: 비-raw 실험과 비교 필터링용 마커. preprocess.run 미사용임을 명시.
    'raw_mode':            True,                # raw임을 명시 (분석/필터링용 마커)
    'target_transform':    TARGET_TRANSFORM,
    'clip_y_extreme':      CLIP_Y_EXTREME,
        # effective_pp_params: raw에선 preprocess.run 미사용 → 빈 dict 박제 (PP 없음 명시).
    'effective_pp_params': {},                  # raw mode — preprocess.run 미사용 (빈 dict로 박제)
        # n_trials: Optuna study에 추가할 trial 수. run_queue.py 외부 타임아웃이 실제 종료 결정.
    'n_trials':            N_TRIALS,
        # n_folds: 5-fold OOF 교차 검증 → 학습셋 전체를 골고루 활용.
    'n_folds':             N_FOLDS,
    'n_jobs':              N_JOBS,
    'n_startup_trials':    N_STARTUP_TRIALS,
    'timeout_sec':         TIMEOUT_SEC,
        # seed_kfold: KFold 분할 seed → 재현성. Optuna sampler는 seed=None으로 다양성 확보.
    'seed_kfold':          SEED,
    'anchor':              CATBOOST_ANCHOR,
}

# anchor를 trial 0으로 강제하려면 study를 먼저 만들어 enqueue → 그 다음 run_hpo가 같은 study(이름·storage)에 이어서
# Optuna HPO 라이브러리 (raw 노트북에서는 cell[4] 안에서 직접 import).
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner
# create_study → enqueue_anchor → run_hpo 순서 중요: anchor가 trial 0이 되려면 study가 먼저 생성돼야 함.
_study = optuna.create_study(
    direction='minimize',
    study_name=EXP_ID,
    storage=f'sqlite:///{DB_PATH}',
    load_if_exists=RESUME,                                                                          # RESUME 시 기존 study 이어 받음
    sampler=TPESampler(seed=None, multivariate=True, group=True, n_startup_trials=N_STARTUP_TRIALS),  # TPE: HP 결합 분포 학습, 조건부 축 skip, run마다 다양성
    pruner=MedianPruner(n_warmup_steps=10),                                                          # 처음 10 step은 안 자르고, 그 후 중앙값보다 나쁘면 가지치기
)
# enqueue_anchor: 기존 trial 0개일 때만 CATBOOST_ANCHOR를 trial 0으로 강제 → warm-start (RESUME 시 중복 방지).
hpo.enqueue_anchor(_study, CATBOOST_ANCHOR)   # 기존 trial 없을 때만 enqueue (RESUME 시 중복 방지)

# HPO 본체: unit 단위 KFold OOF → unit RMSE를 minimize. val/test RMSE는 매 trial user_attr에 기록
res = hpo.run_hpo(
    xs_train=xs_train,
    ys_train_unit=ys_input['train'],
    feat_cols=feat_cols_clean,
    model_name=MODEL_NAME,
    n_trials=N_TRIALS,
    n_folds=N_FOLDS,
    n_jobs=N_JOBS,
    target_transform_fn=target_transform_fn,
    target_inverse_fn=target_inverse_fn,
    study_name=EXP_ID,
    storage=f'sqlite:///{DB_PATH}',
    resume_study=RESUME,
    timeout=TIMEOUT_SEC,
    user_attrs=study_meta_for_save,
    xs_val=xs_val,   ys_val_unit=ys_input['validation'],
    xs_test=xs_test, ys_test_unit=ys_input['test'],
)
# best_params: OOF RMSE 최소 trial의 HP. study: 나중에 trial 분석·추가 실험 재사용 가능.
# study 저장: trial 이력·best_params 조회용. study_meta_for_save는 best_params.json에도 함께 저장됨.
study                 = res['study']
best_params_for_refit = res['best_params']
study_meta_for_save['hpo_best_value'] = float(res['best_value'])

# trial 0이 anchor 키들을 그대로 갖고 있는지 확인 (enqueue 정상 동작 검증)
# trial 0 anchor 검증: enqueue 정상이면 anchor 키가 trial 0에 그대로 있어야 함.
first_trial_params = study.trials[0].params
anchor_keys_present = {k: first_trial_params.get(k) for k in CATBOOST_ANCHOR if k in first_trial_params}
print(f'\n[HPO 완료] best OOF RMSE = {res["best_value"]:.6f}')
print(f'[검증] trial 0 params (anchor 키만): {anchor_keys_present}')
print(f'best_params = {best_params_for_refit}')

## 6. Best trial 재학습 (K-fold OOF)

In [6]:
# best HP로 5-fold 재학습 → die-level OOF / val / test 예측 (val·test는 fold 평균)
final = hpo.refit_best(
    xs_train=xs_train, xs_val=xs_val, xs_test=xs_test,
    ys_train_unit=ys_input['train'],
    feat_cols=feat_cols_clean,
    model_name=MODEL_NAME,
    best_params=best_params_for_refit,
    n_folds=N_FOLDS,
    n_jobs=N_JOBS,
    target_transform_fn=target_transform_fn,
    target_inverse_fn=target_inverse_fn,
)

# 후처리 이전(mean 집계) unit RMSE를 정답과 정렬해 계산 — train(OOF) / val / test
# unit-level 정답 Series 준비. OOF·val·test 예측을 같은 인덱스로 정렬해 RMSE 계산.
y_true = ys_input['train'].set_index(KEY_COL)[TARGET_COL]
oof_u  = final['oof_pred_unit'].set_index(KEY_COL)['pred'].loc[y_true.index]
# 후처리 이전(mean 집계) OOF RMSE: postprocess_config 적용 전 기준값.
oof_rmse = float(np.sqrt(np.mean((oof_u.values - y_true.values)**2)))

# val·test: fold 평균 예측값으로 계산. 실제 성능 대리 지표 (peek-bias 인지하고 사용).
y_val_true  = ys_input['validation'].set_index(KEY_COL)[TARGET_COL]
val_u       = final['val_pred_unit'].set_index(KEY_COL)['pred'].loc[y_val_true.index]
# val·test RMSE: fold 평균 예측으로 계산 (단독 평가 — combine 전).
val_rmse    = float(np.sqrt(np.mean((val_u.values - y_val_true.values)**2)))

y_test_true = ys_input['test'].set_index(KEY_COL)[TARGET_COL]
test_u      = final['test_pred_unit'].set_index(KEY_COL)['pred'].loc[y_test_true.index]
test_rmse   = float(np.sqrt(np.mean((test_u.values - y_test_true.values)**2)))

print(f'\n[Refit 완료] (original space, postprocess 이전)')
print(f'  OOF  unit RMSE = {oof_rmse:.6f}')
print(f'  val  unit RMSE = {val_rmse:.6f}')
print(f'  test unit RMSE = {test_rmse:.6f}')

[refit fold 1/5] tr_units=20949, vl_units=5238


[refit fold 2/5] tr_units=20949, vl_units=5238


[refit fold 3/5] tr_units=20950, vl_units=5237


[refit fold 4/5] tr_units=20950, vl_units=5237


[refit fold 5/5] tr_units=20950, vl_units=5237

[Refit 완료] (original space, postprocess 이전)
  OOF  unit RMSE = 0.005525
  val  unit RMSE = 0.005733
  test unit RMSE = 0.008430


## 7. 후처리 매트릭스 + 산출물 저장

In [7]:
# 후처리 설정 — die→unit 집계 8종 중 best + zero_clip 임계값 탐색. trees는 log space 안 씀(target_transform='none'), π threshold 없음
POSTPROCESS_CONFIG = {
        # agg_methods: die→unit 집계 방식 후보 8종. 각각 val RMSE를 계산해 최적 방식 선택.
    'agg_methods':      ('mean', 'median', 'max', 'min', 'trimmed_mean', 'weighted', 'Q25', 'Q75'),
        # zero_clip_range: 이 범위 안에서 zero_clip 임계값 탐색 (step 단위). zero-inflated 분포 대응.
    'zero_clip_range':  (0.001, 0.015),
    'zero_clip_step':   0.001,
        # zero_clip_log_space: log1p 변환 없으면 False (raw). 임계값 탐색은 linear scale로.
    'zero_clip_log_space': TARGET_TRANSFORM == 'log1p',
        # use_pi_threshold: Two-Stage 분류 확률 임계값 사용 여부. 단일 모델이라 False.
    'use_pi_threshold': False,
}

# fold_models.pkl + best_params.json + die/unit CSV 6개 저장 (postprocess_config 주면 unit CSV는 튜닝값)
# save_artifacts: die/unit CSV 6개 + fold_models.pkl + best_params.json + study_meta → OUT_DIR.
hpo.save_artifacts(
    refit_result=final,
    xs_train=xs_train, xs_val=xs_val, xs_test=xs_test,
    out_dir=OUT_DIR, exp_id=EXP_ID,
    feature_names=feat_cols_clean,
    extra_feature_name=None,
    y_train_unit=ys_input['train'],
    y_val_unit=ys_input['validation'],
    y_test_unit=ys_input['test'],
        # postprocess_config: die→unit 집계 방식·zero_clip 후처리 설정. unit CSV에 튜닝값 반영.
    postprocess_config=POSTPROCESS_CONFIG,
        # study_meta: best_params.json에 함께 저장 → 재현 시 어떤 PP·HP로 돌렸는지 추적 가능.
    study_meta=study_meta_for_save,
)

# 저장된 파일 목록
# 저장 확인: OUT_DIR 파일 목록·크기 출력.
for f in sorted(os.listdir(OUT_DIR)):
        # KB 단위로 출력: 큰 파일(fold_models.pkl)이 제대로 저장됐는지 확인.
    size_kb = os.path.getsize(os.path.join(OUT_DIR, f)) / 1024
    print(f'  {f:30s}  {size_kb:10,.1f} KB')

# Colab이면 산출물을 zip으로 묶어 로컬 PC로 다운로드
# Colab 환경이면 OUT_DIR를 zip으로 묶어 로컬 다운로드. 로컬 환경(ImportError)이면 pass.
try:
    import google.colab
    from google.colab import files
    import shutil
    _zip_base = os.path.join('/content', f'{MODEL_NAME}_{EXP_ID}_outputs')
    _zip_path = shutil.make_archive(_zip_base, 'zip', OUT_DIR)
    print(f'[zip 생성] {_zip_path} ({os.path.getsize(_zip_path)/1024:.1f} KB)')
    try:
        files.download(_zip_path)
    except Exception as _e:
        from IPython.display import FileLink, display
        print(f'[files.download 실패: {_e}] 아래 링크 클릭')
        display(FileLink(_zip_path))
except ImportError:
    pass

[I 2026-05-09 16:56:57,319] A new study created in memory with name: no-name-aee8dad7-50bf-4c96-9a9e-db5a5fad7ad1


[Position weights / Optuna 50t] best=0.005525, w=[0.073, 0.363, 0.268, 0.295]


[Aggregation] RMSEs: {'mean': 0.005525, 'median': 0.005525, 'max': 0.005538, 'min': 0.005529, 'trimmed_mean': 0.005525, 'weighted': 0.005525, 'Q25': 0.005526, 'Q75': 0.005528}
[Aggregation] best=weighted (0.005525)
[zero_clip] best=0.0010 (0.005525)
[Postprocess] best_agg=mean, pi_th=None, zero_clip=0.001, train_rmse=0.005525, val_rmse=0.005731
  baseline_mean                  val_rmse=0.005733086533373272
  after_agg(mean)                val_rmse=0.005733086533373272
  after_pi_th                    val_rmse=0.005733086533373272
  after_zero_clip                val_rmse=0.005731376358068714
  [decision] aggregation    weighted rejected (val 0.005733 <= 0.005733) -> keep mean
  [decision] zero_clip      0.0010 adopted (val 0.005733 -> 0.005731)


[save_artifacts] C:\Users\Dell5371\Desktop\기업연계프로젝트\4_output\02_reg_single\catboost 저장 완료 (fold_models.pkl + best_params.json + 6 CSV, unit=tuned)
  best_params.json                      11.3 KB
  fold_models.pkl                   51,588.1 KB
  oof_die.csv                        5,583.7 KB
  oof_unit.csv                         933.5 KB
  optuna_jh_reg-catboost-002.db        112.0 KB
  test_die.csv                       1,861.0 KB
  test_unit.csv                        311.2 KB
  val_die.csv                        1,861.3 KB
  val_unit.csv                         311.6 KB
